# Session 2 — From planning to learning

**Reinforcement Learning atelier, Université Lumière Lyon 2**

In Session 1 the agent had the map of the world (`env.unwrapped.P`). **Today we take the map away.**
The agent only interacts through `reset()` and `step()`, and has to learn from trial and error.

| Part | Level | What |
|---|---|---|
| 0 | ✏️ | Check your paper answers (gridworld) |
| 1 | 🟢 | Monte Carlo vs TD on the random walk |
| 2 | 🟢 | Q-learning: train a taxi driver |
| 3 | 🟢 | SARSA vs Q-learning on the cliff |
| 4 | 🔴 | Bonus: the Taxi challenge |
| 5 | 🔴 | Bonus: batch MC vs TD, Expected SARSA, Windy Gridworld with king's moves |

Each `TODO` is followed by a **check cell**. Questions marked ❓ are for discussion with your partner.
Cells marked **🔮 Predict** ask you to write down what you expect *before* running the experiment.

In [ ]:
# On Google Colab: uncomment the next line
# %pip install -q "gymnasium[toy-text]"

In [ ]:
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")   # lets pygame render images without opening a window

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import gymnasium as gym

np.set_printoptions(precision=2, suppress=True)

---
## ✏️ Part 0 — Check your paper answers

Iterative policy evaluation of the **random policy** on the $4\times4$ gridworld of the handout
(states numbered as in the book, the two corners are terminal). The code is provided: run it and compare with your answers to A1–A3.

In [ ]:
SIZE = 4
TERMINAL = [(0, 0), (3, 3)]
MOVES = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
ARROW = {"up": "↑", "down": "↓", "left": "←", "right": "→"}

def next_cell(cell, move):
    i, j = cell[0] + MOVES[move][0], cell[1] + MOVES[move][1]
    return (i, j) if 0 <= i < SIZE and 0 <= j < SIZE else cell      # bumping into a wall: stay

def evaluate_random_policy(gamma=1.0, n_sweeps=1000):
    V = np.zeros((SIZE, SIZE))
    for _ in range(n_sweeps):
        V_new = np.zeros_like(V)
        for i in range(SIZE):
            for j in range(SIZE):
                if (i, j) in TERMINAL:
                    continue
                V_new[i, j] = sum(0.25 * (-1 + gamma * V[next_cell((i, j), m)]) for m in MOVES)
        V = V_new
    return V

def q_value(V, cell, move, gamma=1.0):
    return -1 + gamma * V[next_cell(cell, move)]

V_random = evaluate_random_policy()
print("v_pi for the random policy:\n", V_random.round(1))
print("A1: q(11, down) =", round(q_value(V_random, (2, 3), "down"), 2))
print("A2: q(7, down)  =", round(q_value(V_random, (1, 3), "down"), 2))

print("\nA3: greedy policy with respect to v_pi")
for i in range(SIZE):
    row = []
    for j in range(SIZE):
        if (i, j) in TERMINAL:
            row.append("T ")
            continue
        q = {m: q_value(V_random, (i, j), m) for m in MOVES}
        best = max(q.values())
        row.append("".join(ARROW[m] for m in MOVES if np.isclose(q[m], best)).ljust(2))
    print("  ".join(row))

❓ Change `gamma` to 0.9 in `evaluate_random_policy`. Are the values larger or smaller (in absolute value)? Does the greedy policy change?

---
## 🟢 Part 1 — Monte Carlo vs TD on the random walk

```
 [0]  A   B   C   D   E  [6]
  0 ← 1   2   3   4   5 → 6     start in C (state 3), move left or right with probability 1/2
                                 reward +1 when reaching state 6, 0 otherwise
```

The true values under this random policy are $v_\pi(A..E) = 1/6, 2/6, 3/6, 4/6, 5/6$ (probability of exiting on the right).

### 📝 Theory check
Write the update targets for the value of the state $S_t$:
- **Monte Carlo**: $V(S_t) \leftarrow V(S_t) + \alpha\,\big(\;\underline{\hspace{2cm}}\; - V(S_t)\big)$
- **TD(0)**: $V(S_t) \leftarrow V(S_t) + \alpha\,\big(\;\underline{\hspace{2cm}}\; - V(S_t)\big)$

Which target uses the current estimate $V$ (it *bootstraps*)? Which one can only be computed at the end of the episode?

In [ ]:
class RandomWalk(gym.Env):
    # The random walk of Sutton & Barto, Example 6.2, with the Gymnasium API
    def __init__(self, n_states=5):
        self.n = n_states
        self.observation_space = gym.spaces.Discrete(n_states + 2)
        self.action_space = gym.spaces.Discrete(2)               # 0: left, 1: right

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = (self.n + 1) // 2                           # start in the middle (C)
        return self.state, {}

    def step(self, action):
        self.state += 1 if action == 1 else -1
        terminated = self.state in (0, self.n + 1)
        reward = 1.0 if self.state == self.n + 1 else 0.0
        return self.state, reward, terminated, False, {}

TRUE_V = np.arange(1, 6) / 6

def random_policy_episode(env, rng):
    # Plays one episode with the random policy. Returns a list of (state, reward, next_state, terminated)
    state, _ = env.reset()
    transitions, terminated = [], False
    while not terminated:
        next_state, reward, terminated, truncated, _ = env.step(int(rng.integers(2)))
        transitions.append((state, reward, next_state, terminated))
        state = next_state
    return transitions

def step_size(alpha, state):
    # alpha is either a number (constant step size) or a function of the state (used in the stretch part)
    return alpha(state) if callable(alpha) else alpha

### TODO 1 — The TD(0) update

Update `V[state]` **in place** after one transition. Terminal states have no future: when `terminated` is true, the target is just the reward.

In [ ]:
def td0_update(V, state, reward, next_state, terminated, alpha, gamma=1.0):
    a = step_size(alpha, state)
    # TODO 1: V[state] <- V[state] + a * (TD target - V[state])
    raise NotImplementedError

In [ ]:
# Check
V = np.full(7, 0.5); V[0] = V[6] = 0.0
td0_update(V, 5, 1.0, 6, True, alpha=0.1)
assert np.isclose(V[5], 0.55), V[5]           # 0.5 + 0.1 * (1 - 0.5)
td0_update(V, 4, 0.0, 5, False, alpha=0.1)
assert np.isclose(V[4], 0.505), V[4]          # 0.5 + 0.1 * (0 + 0.55 - 0.5)
print("TODO 1 OK")

### TODO 2 — The Monte Carlo update (every-visit)

At the end of an episode, compute the return $G_t$ of **every** time step and update `V[state]` towards it.

> **Tip.** Go *backwards* through the episode: $G_t = R_{t+1} + \gamma\, G_{t+1}$ (the recursion from Session 1!).

In [ ]:
def mc_update(V, transitions, alpha, gamma=1.0):
    G = 0.0
    # TODO 2: loop over reversed(transitions); for each (state, reward, next_state, terminated):
    #         update the return G, then move V[state] towards G with step size step_size(alpha, state)
    raise NotImplementedError

In [ ]:
# Check: episode C -> D -> E -> exit right
V = np.full(7, 0.5); V[0] = V[6] = 0.0
mc_update(V, [(3, 0.0, 4, False), (4, 0.0, 5, False), (5, 1.0, 6, True)], alpha=0.5)
assert np.allclose(V[3:6], [0.75, 0.75, 0.75]), V[3:6]     # every return is 1
mc_update(V, [(3, 0.0, 2, False), (2, 0.0, 1, False), (1, 0.0, 0, True)], alpha=0.5)
assert np.allclose(V[1:4], [0.25, 0.25, 0.375]), V[1:4]    # every return is 0
print("TODO 2 OK")

### The experiment

We measure the **RMS error** between the estimate and the true values, averaged over 100 independent runs.

**🔮 Predict** (write it down before running!): which method has the lower error after 20 episodes? After 100 episodes? Does a larger step size help?

In [ ]:
def rms_curve(method, alpha, n_episodes=100, n_runs=100, seed=0):
    env = RandomWalk()
    errors = np.zeros(n_episodes)
    for run in range(n_runs):
        rng = np.random.default_rng(seed + run)
        V = np.zeros(7); V[1:6] = 0.5
        a = alpha() if callable(alpha) else alpha           # a fresh step-size schedule for each run
        for episode in range(n_episodes):
            errors[episode] += np.sqrt(np.mean((V[1:6] - TRUE_V) ** 2))
            transitions = random_policy_episode(env, rng)
            if method == "TD":
                for t in transitions:
                    td0_update(V, *t, alpha=a)
            else:
                mc_update(V, transitions, alpha=a)
    return errors / n_runs

plt.figure(figsize=(7, 4))
for alpha in [0.05, 0.1, 0.15]:
    plt.plot(rms_curve("TD", alpha), label=f"TD(0), α={alpha}")
for alpha in [0.01, 0.02, 0.04]:
    plt.plot(rms_curve("MC", alpha), "--", label=f"MC, α={alpha}")
plt.xlabel("episodes"); plt.ylabel("RMS error (avg over 100 runs)"); plt.legend(); plt.title("Random walk: MC vs TD")
plt.show()

❓ **Questions**
1. Was your prediction right? Which method learns faster here?
2. For TD, the largest step size is the best at the beginning but not at the end. Why?
3. Why does MC need a *smaller* step size than TD to work well? *(Think about the variance of the targets.)*

### 🟠 Stretch — Constant step size vs $\alpha = 1/N(s)$ (Robbins–Monro)

The convergence theorems need $\sum_t \alpha_t = \infty$ and $\sum_t \alpha_t^2 < \infty$. A constant $\alpha$ violates the second condition, $\alpha = 1/N(s)$ (one over the number of visits of $s$) satisfies both.

**🔮 Predict**: after 500 episodes, which curve is still going down?

In [ ]:
class SampleAverage:
    # Step size 1/N(s): N(s) counts the updates of state s, so V(s) is the average of its targets
    def __init__(self):
        self.N = np.zeros(7)
    def __call__(self, state):
        self.N[state] += 1
        return 1.0 / self.N[state]

plt.figure(figsize=(7, 4))
plt.plot(rms_curve("MC", 0.02, n_episodes=500, n_runs=50), "--", label="MC, α=0.02")
plt.plot(rms_curve("MC", SampleAverage, n_episodes=500, n_runs=50), "--", label="MC, α=1/N(s)")
plt.plot(rms_curve("TD", 0.1, n_episodes=500, n_runs=50), label="TD(0), α=0.1")
plt.plot(rms_curve("TD", SampleAverage, n_episodes=500, n_runs=50), label="TD(0), α=1/N(s)")
plt.xlabel("episodes"); plt.ylabel("RMS error"); plt.legend(); plt.yscale("log")
plt.show()

❓ **Questions**
1. Which curves stop improving (a plateau), and which ones keep going down? Relate this to the two Robbins–Monro conditions.
2. In *control* (Parts 2–4) we will use a constant $\alpha$ anyway. Why can this be a good idea there? *(Hint: does the target stay the same when the policy changes?)*

---
## 🟢 Part 2 — Q-learning: train a taxi driver

The taxi must pick up a passenger (at R, G, Y or B) and drop them at their destination.
**500 states, 6 actions** (0 south, 1 north, 2 east, 3 west, 4 pick up, 5 drop off).
Rewards: $-1$ per step, $+20$ for a delivery, $-10$ for an illegal pick-up or drop-off. Episodes are cut after 200 steps.

In [ ]:
env = gym.make("Taxi-v4", render_mode="ansi")
state, _ = env.reset(seed=1)
print(env.render())
taxi_row, taxi_col, passenger, destination = env.unwrapped.decode(state)
print(f"state {state}: taxi at ({taxi_row},{taxi_col}), passenger at location {passenger}, destination {destination}")

In [ ]:
def play(env_id, policy, seed=0, max_steps=40, **env_kwargs):
    # Plays one episode with `policy(state) -> action` and returns an animation
    env = gym.make(env_id, render_mode="rgb_array", **env_kwargs)
    state, _ = env.reset(seed=seed)
    frames, total = [env.render()], 0
    for _ in range(max_steps):
        state, reward, terminated, truncated, _ = env.step(policy(state))
        frames.append(env.render()); total += reward
        if terminated or truncated:
            break
    env.close()
    fig, ax = plt.subplots(figsize=(4, 4)); ax.axis("off")
    im = ax.imshow(frames[0]); ax.set_title(f"return = {total:.0f}")
    anim = animation.FuncAnimation(fig, lambda i: [im.set_data(frames[i])], frames=len(frames), interval=200)
    plt.close(fig)
    return HTML(anim.to_jshtml())

rng_demo = np.random.default_rng(0)
play("Taxi-v4", lambda s: int(rng_demo.integers(6)), seed=1)     # before learning: a random taxi driver

### 📝 Theory check
The Q-learning target is $R + \gamma \max_{a'} Q(S', a')$.
- Which policy's value does it estimate: the $\varepsilon$-greedy policy that *chooses the actions*, or the greedy policy?
- Why is Q-learning called **off-policy**?

### TODO 3 — $\varepsilon$-greedy action selection

With probability $\varepsilon$ choose a uniformly random action, otherwise a greedy action. **Break ties at random**:
at the beginning all Q-values are 0, and always picking action 0 would make the agent very stupid.

In [ ]:
def epsilon_greedy(Q, state, epsilon, rng):
    # TODO 3: use rng.random(), rng.integers(...), and np.flatnonzero(Q[state] == Q[state].max()) for the ties
    raise NotImplementedError

In [ ]:
# Check
rng = np.random.default_rng(0)
Q_test = np.array([[0.0, 1.0, 1.0, 0.0]])
greedy = [epsilon_greedy(Q_test, 0, 0.0, rng) for _ in range(1000)]
assert set(greedy) == {1, 2}, "with epsilon = 0, only the best actions (1 and 2) should be chosen, both of them"
explore = [epsilon_greedy(Q_test, 0, 1.0, rng) for _ in range(1000)]
assert set(explore) == {0, 1, 2, 3}, "with epsilon = 1, every action should be chosen"
print("TODO 3 OK")

### TODO 4 — Q-learning

$$Q(S,A) \leftarrow Q(S,A) + \alpha \big( R + \gamma \max_{a'} Q(S',a') - Q(S,A) \big)$$

Complete the loop of one episode. Two details matter:
- if `terminated`, there is no future: the target is just `R`;
- if `truncated` (time limit), the MDP itself is not over: **keep** the bootstrap term.

We choose the next action `next_action` *before* the update, and use it as the action of the next step.
(This does not change Q-learning, and Part 3 will show why it is convenient.)

In [ ]:
def q_learning(env, n_episodes, alpha=0.5, gamma=0.99, epsilon=0.1, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    returns = []
    for episode in range(n_episodes):
        state, _ = env.reset(seed=seed + episode)
        action = epsilon_greedy(Q, state, epsilon, rng)
        done, total = False, 0.0
        while not done:
            # TODO 4: step the environment, choose next_action (epsilon-greedy),
            #         update Q[state, action] with the Q-learning target, then move on:
            #         state, action = next_state, next_action
            #         (do not forget to update `done` and `total`)
            raise NotImplementedError
        returns.append(total)
    return Q, np.array(returns)


def evaluate_greedy(env, Q, n_episodes=100, seed=100_000):
    # Average return of the greedy policy (no exploration) on fixed evaluation episodes
    total = 0.0
    for episode in range(n_episodes):
        state, _ = env.reset(seed=seed + episode)
        done = False
        while not done:
            state, reward, terminated, truncated, _ = env.step(int(np.argmax(Q[state])))
            total += reward
            done = terminated or truncated
    return total / n_episodes

In [ ]:
taxi = gym.make("Taxi-v4")
Q_taxi, returns_taxi = q_learning(taxi, n_episodes=2000, alpha=0.5, gamma=0.99, epsilon=0.1)
score = evaluate_greedy(taxi, Q_taxi)
print(f"greedy policy after 2000 episodes: average return {score:.2f}  (optimal: about 7.9)")

plt.figure(figsize=(7, 3))
plt.plot(np.convolve(returns_taxi, np.ones(50) / 50, mode="valid"))
plt.xlabel("episode"); plt.ylabel("training return (moving avg 50)"); plt.title("Q-learning on Taxi")
plt.show()

# Check
assert score > 7.0, "the greedy policy should be close to optimal after 2000 episodes"
print("TODO 4 OK")

In [ ]:
play("Taxi-v4", lambda s: int(np.argmax(Q_taxi[s])), seed=1)     # after learning

❓ **Questions**
1. The *training* returns stay below the greedy evaluation score, even at the end. Why?
2. Train for only 300 episodes and evaluate the greedy policy. What happens to some evaluation episodes? Why can't the table help in a state it has never visited?

---
## 🟢 Part 3 — SARSA vs Q-learning on the cliff

```
. . . . . . . . . . . .
. . . . . . . . . . . .
. . . . . . . . . . . .
S C C C C C C C C C C G      S: start, G: goal, C: cliff (reward -100, back to S)
```
Reward $-1$ per step. Actions: 0 up, 1 right, 2 down, 3 left.

### 📝 Theory check
SARSA uses the target $R + \gamma\, Q(S', A')$, where $A'$ is the action that will **actually** be taken in $S'$.
Which policy's value does SARSA estimate? Is it on-policy or off-policy?

### TODO 5 — SARSA

Copy your `q_learning` function and change **one line** to get SARSA.

In [ ]:
def sarsa(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0):
    # TODO 5: copy the body of q_learning here and change the target
    raise NotImplementedError

In [ ]:
# Check
cliff = gym.make("CliffWalking-v1")
Q_check, _ = sarsa(cliff, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0)
assert Q_check.shape == (48, 4)
assert Q_check[36].max() > -30, "after 500 episodes SARSA should have found a path to the goal"
print("TODO 5 OK")

**🔮 Predict** (write it down!):
1. Which algorithm falls off the cliff more often **during training**?
2. Which path will each greedy policy take at the end?

In [ ]:
def greedy_path(Q, start=36, goal=47, max_steps=60):
    env = gym.make("CliffWalking-v1")
    state, _ = env.reset()
    path = [state]
    while state != goal and len(path) < max_steps:
        state, *_ = env.step(int(np.argmax(Q[state])))
        path.append(state)
    return path

def draw_paths(paths):
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.add_patch(plt.Rectangle((0.5, 2.5), 10, 1, color="indianred", alpha=0.6))
    ax.text(5.5, 3, "cliff", ha="center", va="center", color="white", fontsize=12)
    for (label, path), color in zip(paths.items(), ["tab:blue", "tab:orange"]):
        rows, cols = np.divmod(np.array(path), 12)
        ax.plot(cols, rows, "-o", color=color, label=f"{label} ({len(path) - 1} steps)", lw=2, ms=4)
    ax.text(0, 3, "S", ha="center", va="center", fontsize=14, weight="bold")
    ax.text(11, 3, "G", ha="center", va="center", fontsize=14, weight="bold")
    ax.set_xlim(-0.5, 11.5); ax.set_ylim(3.5, -0.5); ax.set_xticks(range(12)); ax.set_yticks(range(4))
    ax.grid(True, alpha=0.3); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
    plt.show()

def compare(n_runs=20, n_episodes=500, epsilon=0.1, alpha=0.5):
    curves = {"SARSA": np.zeros(n_episodes), "Q-learning": np.zeros(n_episodes)}
    mean_Q = {name: np.zeros((48, 4)) for name in curves}      # Q-tables averaged over the runs
    for run in range(n_runs):
        for name, algo in [("SARSA", sarsa), ("Q-learning", q_learning)]:
            Q, returns = algo(cliff, n_episodes, alpha=alpha, gamma=1.0, epsilon=epsilon, seed=1000 * run)
            curves[name] += returns / n_runs
            mean_Q[name] += Q / n_runs
    plt.figure(figsize=(7, 4))
    for name, curve in curves.items():
        plt.plot(np.convolve(curve, np.ones(10) / 10, mode="valid"), label=name)
    plt.ylim(-100, 0); plt.xlabel("episode"); plt.ylabel("sum of rewards during episode")
    plt.title(f"CliffWalking, ε = {epsilon}"); plt.legend(); plt.show()
    draw_paths({name: greedy_path(Q) for name, Q in mean_Q.items()})

compare()

❓ **Questions**
1. Was your prediction right? Explain the two paths with the on-policy / off-policy distinction.
2. Q-learning finds the *optimal* path, yet collects *less* reward during training. Which algorithm would you rather use to train a real robot next to a real cliff?
3. Run `compare(epsilon=0.0)`. What happens, and why? *(Is the agent still exploring?)*

### 🔴 Bonus — GLIE in practice

The convergence theorem for SARSA needs a **GLIE** policy: explore infinitely often, but become greedy in the limit.
With $\varepsilon$ decaying to 0, SARSA should end up on the optimal path along the edge. Does it?

In [ ]:
def sarsa_decaying_epsilon(env, n_episodes, alpha=0.5, eps_start=0.1, seed=0):
    # Same as SARSA, with epsilon decreasing linearly from eps_start to 0 over 80% of the episodes
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    for episode in range(n_episodes):
        epsilon = max(0.0, eps_start * (1 - episode / (0.8 * n_episodes)))
        state, _ = env.reset(seed=seed + episode)
        action = epsilon_greedy(Q, state, epsilon, rng)
        done = False
        while not done:
            next_state, reward, terminated, truncated, _ = env.step(action)
            next_action = epsilon_greedy(Q, next_state, epsilon, rng)
            Q[state, action] += alpha * (reward + Q[next_state, next_action] * (not terminated) - Q[state, action])
            state, action = next_state, next_action
            done = terminated or truncated
    return Q

for seed in range(3):
    Q_glie = sarsa_decaying_epsilon(cliff, n_episodes=3000, seed=seed)
    print(f"seed {seed}: greedy path of {len(greedy_path(Q_glie)) - 1} steps (optimal: 13, safe path: 17)")

❓ The theorem says SARSA converges to $q_*$. Check its conditions one by one (GLIE, Robbins–Monro). Which ones does our experiment satisfy? Why does the agent still take the safe path?

---
## 🔴 Part 4 — Bonus: the Taxi challenge

**Goal:** the best greedy score on Taxi with **only 500 training episodes**.
Baseline (`q_learning` with $\alpha=0.5, \gamma=0.99, \varepsilon=0.1$): about $-46$. Optimal policy: $7.9$.

**Rules**
- exactly `n_episodes = 500` calls to `env.reset()` for training, only `reset()` and `step()` (no `env.unwrapped.P`!);
- your function returns a Q-table; the score is the greedy evaluation below (3 training seeds, 100 fixed evaluation episodes each).

**Levers to try:** $\alpha$, $\gamma$, $\varepsilon$ and how it changes over time, the initial Q-values, ... and any idea from the lecture.
Remember the questions of Part 2: why does the greedy policy fail after a short training?

In [ ]:
def my_agent(env, n_episodes, seed):
    # Change anything you want here (you can write a new training loop)
    Q, _ = q_learning(env, n_episodes, alpha=0.5, gamma=0.99, epsilon=0.1, seed=seed)
    return Q

def challenge_score(agent, n_episodes=500):
    env = gym.make("Taxi-v4")
    return np.mean([evaluate_greedy(env, agent(env, n_episodes, seed)) for seed in (0, 1, 2)])

print(f"your score: {challenge_score(my_agent):.2f}   (baseline about -46, optimal 7.9)")

❓ Which lever helped most? Can you explain *why* with the theory (deterministic environment, discounting, exploration)?

---
## 🔴 Part 5 — Bonus

### 5a. Batch MC vs batch TD on the AB example (handout Part C)
Replay the 8 episodes many times with a small step size, and compare the final $V(A), V(B)$ with your paper answers.
States: 0 = A, 1 = B, 2 = terminal.

In [ ]:
AB_EPISODES = [[(0, 0.0, 1, False), (1, 0.0, 2, True)]] + [[(1, 1.0, 2, True)]] * 6 + [[(1, 0.0, 2, True)]]

def batch(method, n_sweeps=5000, alpha=0.01):
    V = np.zeros(3)
    # TODO 6: for n_sweeps sweeps, apply mc_update to every episode (method == "MC"),
    #         or td0_update to every transition of every episode (method == "TD")
    raise NotImplementedError
    return V

for method in ["MC", "TD"]:
    V = batch(method)
    print(f"batch {method}: V(A) = {V[0]:.3f}, V(B) = {V[1]:.3f}")

### 5b. Expected SARSA
Replace $Q(S', A')$ in the SARSA target by its **expectation** under the $\varepsilon$-greedy policy:
$\sum_{a'} \pi(a' \mid S')\, Q(S', a')$. Write `expected_sarsa` and compare it with SARSA and Q-learning on the cliff
(training rewards and final path). Sutton & Barto, Section 6.6.

In [ ]:
def expected_sarsa(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0):
    # TODO 7 (bonus)
    raise NotImplementedError

In [ ]:
Q_es, returns_es = expected_sarsa(cliff, 500)
print("average reward over the last 100 training episodes:", returns_es[-100:].mean())
draw_paths({"Expected SARSA": greedy_path(Q_es)})

### 5c. Windy Gridworld: change the world, not the algorithm
Sutton & Barto, Example 6.5 and Exercise 6.9. A wind pushes the agent **up** by 0, 1 or 2 cells depending on the column.
With the 4 usual moves the shortest path takes 15 steps. Add the **king's moves** (the 4 diagonals): how short does it get?
Your `sarsa` function works unchanged on this new environment.

In [ ]:
class WindyGridworld(gym.Env):
    WIND = [0, 0, 0, 1, 1, 1, 2, 2, 1, 0]
    HEIGHT, WIDTH = 7, 10
    START, GOAL = (3, 0), (3, 7)

    def __init__(self, moves):
        self.moves = moves                                     # list of (d_row, d_col)
        self.observation_space = gym.spaces.Discrete(self.HEIGHT * self.WIDTH)
        self.action_space = gym.spaces.Discrete(len(moves))

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.pos = self.START
        return self.pos[0] * self.WIDTH + self.pos[1], {}

    def step(self, action):
        i, j = self.pos
        di, dj = self.moves[action]
        i = min(max(i + di - self.WIND[j], 0), self.HEIGHT - 1)     # the wind of the current column pushes up
        j = min(max(j + dj, 0), self.WIDTH - 1)
        self.pos = (i, j)
        terminated = self.pos == self.GOAL
        return i * self.WIDTH + j, -1.0, terminated, False, {}

MOVES_4 = [(-1, 0), (1, 0), (0, -1), (0, 1)]
MOVES_8 = MOVES_4 + []     # TODO 8 (bonus): add the 4 diagonal moves

plt.figure(figsize=(7, 4))
for label, moves in [("4 moves", MOVES_4), ("king's moves", MOVES_8)]:
    windy = WindyGridworld(moves)
    Q_w, returns_w = sarsa(windy, n_episodes=300, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0)
    plt.plot(np.cumsum(-returns_w), np.arange(1, 301), label=label)      # -return = number of steps
    print(f"{label}: the last 50 episodes take {-returns_w[-50:].mean():.1f} steps on average (with exploration)")
plt.xlabel("time steps"); plt.ylabel("episodes"); plt.legend(); plt.title("Windy Gridworld (S&B Figure 6.3)")
plt.show()

❓ What changed between the two experiments: the algorithm or the MDP? Would adding a 9th action "stay" help?

---
## Wrap-up

| | Needs a model? | Bootstraps? | Target |
|---|---|---|---|
| DP (Session 1, Part 0) | yes | yes | $\sum_{s'} \mathcal{P}^a_{ss'}(r + \gamma V(s'))$ |
| Monte Carlo | no | no | $G_t$ |
| TD(0) | no | yes | $R + \gamma V(S')$ |
| SARSA | no | yes | $R + \gamma Q(S', A')$ |
| Q-learning | no | yes | $R + \gamma \max_{a'} Q(S', a')$ |

**Session 3:** Taxi has 500 states, so a table works. In LunarLander the state is 8 real numbers: a table is impossible.
We replace the table by a neural network: **Deep Q-Networks**.